# Using MiniMax API Endpoints

This notebook extends the API-backed generation patterns from Chapters 4 and 7 with MiniMax. It covers the OpenAI-compatible chat completions interface and the Anthropic-compatible messages interface in both supported regions.

Model and endpoint selection stay in one executable configuration so the request cells do not need region-specific changes.

### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

Uncomment and run the following cell when the compatible client packages are not already installed.

In [ ]:
# %%capture
# !pip install openai anthropic

## Model Configuration

| Model | Context window | Input modalities | Thinking modes |
|-------|---------------:|------------------|----------------|
| MiniMax-M3 | 1000000 | text, image, video | adaptive, disabled |
| MiniMax-M2.7 | 204800 | text | always_on |

Pricing below is in USD per million tokens.

| Model | Input | Output | Cache read | Cache write |
|-------|------:|-------:|-----------:|------------:|
| MiniMax-M3 | 0.6 | 2.4 | 0.12 | n/a |
| MiniMax-M2.7 | 0.3 | 1.2 | 0.06 | 0.375 |

In [ ]:
import json
import os

from anthropic import Anthropic
from openai import OpenAI

MINIMAX_MODELS = json.loads("{\"MiniMax-M3\":{\"context_window\":1000000,\"pricing_usd_per_million_tokens\":{\"input\":0.6,\"output\":2.4,\"cache_read\":0.12,\"cache_write\":null},\"input_modalities\":[\"text\",\"image\",\"video\"],\"thinking\":[\"adaptive\",\"disabled\"]},\"MiniMax-M2.7\":{\"context_window\":204800,\"pricing_usd_per_million_tokens\":{\"input\":0.3,\"output\":1.2,\"cache_read\":0.06,\"cache_write\":0.375},\"input_modalities\":[\"text\"],\"thinking\":[\"always_on\"]}}")
MINIMAX_ENDPOINTS = json.loads("{\"global_en\":{\"openai_base_url\":\"https://api.minimax.io/v1\",\"anthropic_base_url\":\"https://api.minimax.io/anthropic\",\"docs_root\":\"https://platform.minimax.io/docs\"},\"cn_zh\":{\"openai_base_url\":\"https://api.minimaxi.com/v1\",\"anthropic_base_url\":\"https://api.minimaxi.com/anthropic\",\"docs_root\":\"https://platform.minimaxi.com/docs\"}}")

region_name = os.environ.get("MINIMAX_REGION", "global_en")
model_id = os.environ.get("MINIMAX_MODEL", "MiniMax-M3")

if region_name not in MINIMAX_ENDPOINTS:
    raise ValueError(f"Unsupported MiniMax region: {region_name}")
if model_id not in MINIMAX_MODELS:
    raise ValueError(f"Unsupported MiniMax model: {model_id}")

endpoint = MINIMAX_ENDPOINTS[region_name]
model_config = MINIMAX_MODELS[model_id]

## Chat Completions Interface

The selected region supplies the matching OpenAI-compatible base URL.

In [ ]:
openai_client = OpenAI(
    api_key=os.environ["MINIMAX_API_KEY"],
    base_url=endpoint["openai_base_url"],
)

openai_response = openai_client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "user", "content": "Explain context windows in one sentence."}
    ],
)
print(openai_response.choices[0].message.content)

## Messages Interface

The same model and region selection can be used with the Anthropic-compatible base URL.

In [ ]:
anthropic_client = Anthropic(
    api_key=os.environ["MINIMAX_API_KEY"],
    base_url=endpoint["anthropic_base_url"],
)

anthropic_response = anthropic_client.messages.create(
    model=model_id,
    max_tokens=256,
    messages=[
        {"role": "user", "content": "Explain context windows in one sentence."}
    ],
)
print(anthropic_response.content[0].text)

## Switching Configuration

Set MINIMAX_REGION to global_en or cn_zh and MINIMAX_MODEL to MiniMax-M3 or MiniMax-M2.7, then rerun the configuration and request cells. The endpoint dictionary also exposes the matching documentation root.